In [ ]:
from skimage.feature import graycomatrix, graycoprops

#Funcion para calcular los descriptores de cada canal
def obtencionCaracteristicas (img):
    glcm = graycomatrix(rojo, distances=[1], angles=[0], levels=256, symmetric=True, normed=True)
    contraste = graycoprops(glcm, prop="contrast")[0,0]
    energia = graycoprops(glcm, prop= "energy")[0,0]
    ASM = graycoprops(glcm, prop = "ASM")[0,0]
    homogeneidad = graycoprops(glcm, prop= "homogeneity")[0,0]
    correlacion = graycoprops(glcm, prop= "correlation")[0,0]
    disimilaridad = graycoprops(glcm, prop= "dissimilarity")[0,0]
    
    return contraste, energia, ASM, homogeneidad, correlacion, disimilaridad 


In [50]:
import cv2
import pandas as pd
from pathlib import Path
from collections import defaultdict

CARPETA = Path("BD")

contador = defaultdict(int)   # {clase: n_imágenes_vistas}
filas = []

for ruta in sorted(CARPETA.rglob("*.jpg")):
    img = cv2.imread(str(ruta))
    if img is None:
        print(f"⚠️  No se pudo leer: {ruta}")
        continue

    clase = ruta.parent.name
    contador[clase] += 1
    img_id = f"{clase}_img{contador[clase]:02d}"   # textura_01_img01

    azul, verde, rojo = cv2.split(img)
    fila = {"id": img_id}

    for nombre, canal in zip(("azul", "verde", "rojo"), (azul, verde, rojo)):
        contraste, energia, ASM, homogeneidad, correlacion, disimilaridad = \
            obtencionCaracteristicas(canal)

        fila[f"{nombre}_contraste"]     = contraste
        fila[f"{nombre}_energia"]       = energia
        fila[f"{nombre}_ASM"]           = ASM
        fila[f"{nombre}_homogeneidad"]  = homogeneidad
        fila[f"{nombre}_correlacion"]   = correlacion
        fila[f"{nombre}_disimilaridad"] = disimilaridad

    filas.append(fila)

df = pd.DataFrame(filas)

print(df[["id", "clase"]].head(10))
print(df.shape)
print(df["clase"].value_counts())

df.to_csv("Caracteristicas.csv", index=False)
print("\n✅ Guardado en Caracteristicas.csv")

KeyError: "['clase'] not in index"

### Referencias

1. https://scikit--image-org.translate.goog/docs/stable/auto_examples/features_detection/plot_glcm.html?_x_tr_sl=en&_x_tr_tl=es&_x_tr_hl=es&_x_tr_pto=sge

2. https://scikit--image-org.translate.goog/docs/stable/api/skimage.feature.html?_x_tr_sl=en&_x_tr_tl=es&_x_tr_hl=es&_x_tr_pto=tc#skimage.feature.graycoprops